# ANPR End-to-End (Colab Notebook)

This notebook installs dependencies, downloads dataset sources, optionally trains YOLO on Colab GPU, and exports artifacts for laptop use.

In [12]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/ayannotfound/ANPR.git'
DEFAULT_CLONE_DIR = Path('/content/ANPR')

def looks_like_repo(path: Path) -> bool:
    return (path / 'requirements.txt').exists() and (path / 'main.py').exists()

def resolve_repo_root() -> Path | None:
    cwd = Path.cwd()
    candidates = [cwd, DEFAULT_CLONE_DIR]
    candidates.extend(cwd.parents)
    for c in candidates:
        if looks_like_repo(c):
            return c
    return None

repo_root = resolve_repo_root()
if repo_root is None:
    print('Repo not found in kernel filesystem. Cloning now...')
    if DEFAULT_CLONE_DIR.exists() and not looks_like_repo(DEFAULT_CLONE_DIR):
        subprocess.run(['rm', '-rf', str(DEFAULT_CLONE_DIR)], check=False)
    if not DEFAULT_CLONE_DIR.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(DEFAULT_CLONE_DIR)], check=True)
    repo_root = DEFAULT_CLONE_DIR

if not looks_like_repo(repo_root):
    raise FileNotFoundError('Repository still not found after clone attempt.')

os.chdir(repo_root)
print('Repo root:', repo_root)

# Colab-first model: use kernel interpreter directly (no venv).
selected_python = sys.executable
os.environ['ANPR_PYTHON'] = selected_python

subprocess.run([selected_python, '-m', 'pip', 'install', '--upgrade', 'pip', 'setuptools', 'wheel'], check=True)
subprocess.run([selected_python, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)

print('Python for notebook steps:', os.environ['ANPR_PYTHON'])

Repo root: /content/ANPR
Python for notebook steps: /usr/bin/python3


: 

## Dataset Key at Runtime

Roboflow API key link: https://app.roboflow.com/settings/api

In [13]:
import os
from getpass import getpass

rf_key = getpass('Enter ROBOFLOW_API_KEY (leave blank to skip Roboflow source): ').strip()
if rf_key:
    os.environ['ROBOFLOW_API_KEY'] = rf_key
    print('Roboflow key saved in this runtime session.')
else:
    print('No key entered. download_dataset.py will proceed with HF only.')

Roboflow key saved in this runtime session.


In [15]:
import os
import sys
from pathlib import Path
from download_dataset import build_dataset

if not (Path('download_dataset.py').exists() and Path('main.py').exists()):
    raise FileNotFoundError('Run Cell 2 first to set repo root correctly.')

selected_python = os.environ.get('ANPR_PYTHON', sys.executable)
print('Using interpreter:', selected_python)

include_hf = True
include_roboflow = True
roboflow_version = 1
clean_temp = True
roboflow_api_key = os.getenv('ROBOFLOW_API_KEY')

print('=' * 72)
print('ANPR Dataset Builder')
print('=' * 72)
print(f'Sources: HF={include_hf}, Roboflow Indian={include_roboflow}')

yaml_path = build_dataset(
    include_hf=include_hf,
    include_roboflow=include_roboflow,
    roboflow_api_key=roboflow_api_key,
    roboflow_version=roboflow_version,
    clean_temp=clean_temp,
)

print('\nUse this for training:')
print(f'  python train.py  # DATA_YAML already points to: {yaml_path.as_posix()}')

Using interpreter: /usr/bin/python3
ANPR Dataset Builder
Sources: HF=True, Roboflow Indian=True
[HF  ] Downloading Hugging Face dataset...
[HF  ] datasets path failed: Dataset scripts are no longer supported, but found license-plate-object-detection.py
[RF  ] Downloading Indian Roboflow dataset...
loading Roboflow workspace...
loading Roboflow project...
[RF  ] SDK download returned empty dataset. Falling back to API export link...
[MERGE] rf_india: train=583, val=167, test=83
[DONE] Merged dataset -> data/license_plates/data.yaml (train=583, val=167, test=83)

Use this for training:
  python train.py  # DATA_YAML already points to: data/license_plates/data.yaml


## Training (Only If Needed)

Training is slow and expensive. Keep this disabled unless you need a better detector.

In [16]:
import os
import subprocess
import sys
from pathlib import Path

TRAIN_IF_NEEDED = False
selected_python = os.environ.get('ANPR_PYTHON', sys.executable)

if TRAIN_IF_NEEDED:
    subprocess.run([selected_python, 'train.py'], check=True)
else:
    print('Training skipped (default).')

Training skipped (default).


In [ ]:
import os
import shutil
from pathlib import Path

if not os.path.exists('main.py'):
    raise FileNotFoundError('Run Cell 2 first to set repo root correctly.')

# Prepare a single folder with artifacts you can download to laptop.
artifact_dir = Path('/content/anpr_artifacts')
artifact_dir.mkdir(parents=True, exist_ok=True)

# Candidate weight files in priority order.
weight_candidates = [
    Path('runs/detect/license_plate_detector/weights/best.pt'),
    Path('runs/detect/license_plate_detector/weights/last.pt'),
    Path('yolo26n.pt'),
    Path('yolov8n.pt'),
]

copied = []
for src in weight_candidates:
    if src.exists():
        dst = artifact_dir / src.name
        shutil.copy2(src, dst)
        copied.append(dst.name)

# Also copy dataset yaml if available, useful on laptop for retraining or checks.
yaml_path = Path('data/license_plates/data.yaml')
if yaml_path.exists():
    shutil.copy2(yaml_path, artifact_dir / 'data.yaml')
    copied.append('data.yaml')

if not copied:
    raise FileNotFoundError('No artifacts found. Train first or ensure base weights exist in repo root.')

zip_base = Path('/content/anpr_artifacts')
zip_path = shutil.make_archive(str(zip_base), 'zip', root_dir=artifact_dir)

print('Artifacts prepared:', ', '.join(copied))
print('Download this file to your laptop:', zip_path)
print('')
print('Laptop steps after download:')
print('1) Unzip artifacts into your local ANPR repo.')
print('2) Put plate weights at runs/detect/license_plate_detector/weights/best.pt')
print('   (or update PLATE_MODEL_PATH in anpr/pipeline_core.py).')
print('3) Run: python main.py')
print('4) Open: http://localhost:8000')

try:
    from google.colab import files
    files.download(zip_path)
except Exception:
    print('Auto-download unavailable in this environment.')
    print('Manually download from Colab Files panel:', zip_path)

ERROR:pyngrok.process.ngrok:t=2026-03-15T13:11:17+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
ERROR:pyngrok.process.ngrok:t=2026-03-15T13:11:17+0000 lvl=eror msg="session closing" obj=tunnels.session err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
ERROR:pyngrok.process.ngrok:t=2026-03-15T13:11:17+0000 lvl=eror msg="terminating with error" obj=app err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your aut

Ngrok tunnel was not created.
Reason: The ngrok process errored on start: authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n.
If needed, set auth token first: from pyngrok import conf; conf.get_default().auth_token = "<token>"


RuntimeError: asyncio.run() cannot be called from a running event loop